# Stage 1a 模型测试集可视化（CPU 推理）

本 notebook 用于**调试与定性检查**：在 OAM-TCD 官方 test 集（训练管线从未评估过的封存集）上，用 CPU 推理可视化 staged checkpoint `exp-stage1a-oamtcd-y11n-r1` 的分割效果。不计算任何指标，不发布任何产物。

## 前置条件
- 凭据：仓库根目录 `.local/credentials.env` 中已配置 `HF_TOKEN`（模式 600）。
- 环境：当前 venv 需安装 jupyter 内核，首次使用请执行一次 `uv add --dev jupyter`。
- 硬件：无 GPU 即可运行（torch cu126 wheel 在无 GPU 机器上自动回退 CPU），20 张图约需 1–3 分钟。

## 数据与模型说明
- 模型：单类 `tree-crown`（class id 0），`yolo11n-seg`，imgsz 1280，staged 于 `zyzh0/tree-crown-yolo11-seg-staging`（tag `exp-stage1a-oamtcd-y11n-r1`，`deployable: false`）。
- 测试图像：OAM-TCD 源 test 分片共 439 行，训练管线（`prepare_oamtcd.py`）实际写入 417 张；本 notebook 重现同样的丢弃与涂黑逻辑，保证所见即"模型训练时所见"。
- 注意：训练管线会将冠层（canopy，category 1）区域涂黑；notebook 重建训练时图像（含涂黑）后再做推理与对比。

In [ ]:
%matplotlib inline

# --- imports ---
import gc
import hashlib
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

from prepare_oamtcd import (
    DROP_CATEGORY_IDS,
    KEEP_CATEGORY_IDS,
    REPO_ID,
    REVISION,
    _encode_rgb_jpeg,
    _extract_image_bytes,
    _redact_canopy_regions,
    annotations_to_instances,
    segmentation_to_yolo,
)
from publish import _load_token_from_env

# --- repository root (works regardless of the notebook's cwd) ---
cwd = Path.cwd()
REPO_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# --- editable parameters ---
N_IMAGES = 20
SEED = 0
IMGSZ = 1280
CONF = 0.25
DEVICE = "cpu"
SAVE_PNG = False
MAX_DET = 50  # memory guard: one full-res mask is 2048x2048 float32 (~16.8 MB)
DISPLAY_SIZE = 512  # grid thumbnail size (source images are 2048x2048)

# --- staging constants (authoritative source: .agents/docs/experiments/oamtcd-stage1a-r1.md) ---
STAGING_REPO_ID = "zyzh0/tree-crown-yolo11-seg-staging"
STAGING_TAG = "exp-stage1a-oamtcd-y11n-r1"
CHECKPOINT_PATH = "artifacts/exp-stage1a-oamtcd-y11n-r1/model.pt"
CHECKPOINT_SHA256 = "83e2a1c11a8ccfd41207bbd127f4ea69eac270d1dcb54039ec4e71211aef73b9"
TEST_SHARD = "data/test-00000-of-00001.parquet"

# --- HF token (never printed) ---
token = _load_token_from_env(str(REPO_ROOT / ".local" / "credentials.env"))

In [ ]:
def sha256_of(path: Path) -> str:
    """Streaming SHA256 of a file (same intent as publish._sha256)."""
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


checkpoint_path = Path(
    hf_hub_download(STAGING_REPO_ID, filename=CHECKPOINT_PATH, revision=STAGING_TAG, token=token)
)
actual_sha = sha256_of(checkpoint_path)
if actual_sha != CHECKPOINT_SHA256:
    raise RuntimeError(
        f"SHA256 mismatch for {checkpoint_path.name}: "
        f"expected {CHECKPOINT_SHA256}, got {actual_sha}. "
        "Refusing to run inference on an unverified checkpoint."
    )
print("checkpoint verified:", checkpoint_path)

In [ ]:
def is_trainable_row(row: dict) -> bool:
    """Exact mirror of write_image_and_labels' keep/drop decision.

    prepare_oamtcd.py drops a row only when it has canopy annotations but no
    individual-tree annotations (canopy-only), or when tree annotations exist
    but none convert to a valid polygon. Rows with NO annotations at all are
    KEPT as background/negative samples (empty label files) - the training
    pipeline supervises them as empty images, so they must stay visible here.
    """
    annotations = annotations_to_instances(str(row["coco_annotations"]))
    tree_annotations = [a for a in annotations if a.get("category_id") in KEEP_CATEGORY_IDS]
    has_canopy = any(a.get("category_id") in DROP_CATEGORY_IDS for a in annotations)
    if has_canopy and not tree_annotations:
        return False  # canopy-only rows are dropped by the pipeline
    if not tree_annotations:
        return True  # no annotations at all: kept as background/negative
    width, height = int(row["width"]), int(row["height"])
    return any(segmentation_to_yolo(a["segmentation"], width, height) is not None for a in tree_annotations)


parquet_path = Path(
    hf_hub_download(REPO_ID, filename=TEST_SHARD, revision=REVISION, token=token, repo_type="dataset")
)
source_rows = pl.read_parquet(parquet_path).to_dicts()
trainable_rows = [r for r in source_rows if is_trainable_row(r)]
print(f"source test rows: {len(source_rows)}; training-pipeline-accepted: {len(trainable_rows)}")

if len(trainable_rows) < N_IMAGES:
    raise RuntimeError(f"Only {len(trainable_rows)} trainable test rows available, fewer than N_IMAGES={N_IMAGES}.")

sampled = pl.DataFrame(trainable_rows).sample(n=N_IMAGES, seed=SEED).to_dicts()
print("sampled image_ids:", [r["image_id"] for r in sampled])

# Release the 439 source frames (2048x2048 JPEG bytes ~= 1.3 GB) before inference.
del source_rows, trainable_rows
gc.collect()


In [ ]:
def reconstruct_training_image(row: dict) -> np.ndarray:
    """Rebuild the exact RGB image the model was supervised on.

    Extract source bytes, redact canopy regions to black when present, and
    re-encode as a quality-95 RGB JPEG (the byte-identical path used by
    write_image_and_labels).
    """
    img_bytes = _extract_image_bytes(row)
    if img_bytes is None:
        raise RuntimeError(f"Failed to extract image bytes for image_id={row['image_id']}")

    annotations = annotations_to_instances(str(row["coco_annotations"]))
    tree_annotations = [a for a in annotations if a.get("category_id") in KEEP_CATEGORY_IDS]
    canopy_annotations = [a for a in annotations if a.get("category_id") in DROP_CATEGORY_IDS]

    if canopy_annotations:
        img_bytes = _redact_canopy_regions(img_bytes, canopy_annotations, tree_annotations)
    else:
        img_bytes = _encode_rgb_jpeg(img_bytes)
    if img_bytes is None:
        raise RuntimeError(f"Failed to rebuild image bytes for image_id={row['image_id']}")

    image_bgr = cv2.imdecode(np.frombuffer(img_bytes, dtype=np.uint8), cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise RuntimeError(f"Failed to decode rebuilt image for image_id={row['image_id']}")
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


def gt_polygons(row: dict) -> list[np.ndarray]:
    """Training-time GT polygons (category 2 only) in pixel coordinates."""
    width, height = int(row["width"]), int(row["height"])
    annotations = annotations_to_instances(str(row["coco_annotations"]))
    tree_annotations = [a for a in annotations if a.get("category_id") in KEEP_CATEGORY_IDS]
    polygons = []
    for a in tree_annotations:
        coords = segmentation_to_yolo(a["segmentation"], width, height)
        if coords is None:
            continue
        pts = np.array(coords, dtype=np.float64).reshape(-1, 2)
        pts[:, 0] *= width
        pts[:, 1] *= height
        polygons.append(pts)
    return polygons


# Rebuild all sampled images once; reuse for display and inference.
images = [reconstruct_training_image(r) for r in sampled]
print("rebuilt images:", len(images))

In [ ]:
# Single-class model: names may print generically but class id 0 is tree-crown.
# Memory-conscious loop: predict ONE image at a time, render the overlay
# immediately, then drop the result. Each result can hold up to MAX_DET
# full-res (2048x2048 float32) masks; keeping 20 results alive at once is
# what OOMs an 8 GB machine.
model = YOLO(str(checkpoint_path))
pred_overlays = []
for i, img in enumerate(images):
    result = model.predict(
        source=img,
        device=DEVICE,
        imgsz=IMGSZ,
        conf=CONF,
        max_det=MAX_DET,
        verbose=False,
    )[0]
    pred_bgr = result.plot()  # overlay rendered at full res, BGR
    pred_overlays.append(pred_bgr[..., ::-1])
    del result
    gc.collect()
print("predictions:", len(pred_overlays))

In [ ]:
def thumbnail(img: np.ndarray) -> np.ndarray:
    """Downscale for display (memory + readability); INTER_AREA for decimation."""
    if max(img.shape[:2]) <= DISPLAY_SIZE:
        return img
    scale = DISPLAY_SIZE / max(img.shape[:2])
    size = (max(1, round(img.shape[1] * scale)), max(1, round(img.shape[0] * scale)))
    return cv2.resize(img, size, interpolation=cv2.INTER_AREA)


fig, axes = plt.subplots(N_IMAGES, 3, figsize=(15, 5 * N_IMAGES))
axes = np.atleast_2d(axes)
for i, (row, overlay) in enumerate(zip(sampled, pred_overlays)):
    thumb = thumbnail(images[i])
    scale = thumb.shape[1] / images[i].shape[1]  # display scale for polygon coords

    # Column 1: training-time input (canopy regions already blacked out).
    axes[i, 0].imshow(thumb)
    axes[i, 0].set_title(f"input {row['image_id']}")
    axes[i, 0].axis("off")

    # Column 2: ground-truth polygons (category 2 only, training-time conversion).
    axes[i, 1].imshow(thumb)
    for pts in gt_polygons(row):
        axes[i, 1].plot(pts[:, 0] * scale, pts[:, 1] * scale, color="lime", linewidth=1.2)
    axes[i, 1].set_title("GT (tree-crown)")
    axes[i, 1].axis("off")

    # Column 3: prediction overlay rendered at inference time (BGR -> RGB).
    axes[i, 2].imshow(thumbnail(overlay))
    axes[i, 2].set_title("prediction")
    axes[i, 2].axis("off")

fig.tight_layout()
plt.show()

if SAVE_PNG:
    out_dir = REPO_ROOT / "runs" / "visualize" / STAGING_TAG
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / "grid.png", dpi=120)
    print("saved:", out_dir / "grid.png")

## 清理 HF 缓存（可选）

模型与 parquet 会缓存在 `~/.cache/huggingface`。如需释放空间：

- 查看缓存：`hf cache list`
- 删除 checkpoint 缓存：`hf cache rm zyzh0/tree-crown-yolo11-seg-staging`
- 删除数据集缓存：`hf cache rm restor/tcd`
- 清理所有未引用版本：`hf cache prune`